In [3]:
 from google.colab import drive
 drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -q pydicom nibabel numpy pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:00


In [6]:

# Root folder that contains all case folders
ROOT_DIR = "/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Real Patients"

     #This version assumes each case looks like this:
            # case_001/
                      # PT/
                           # ... PET DICOM files ...
                      # mask/
                           # lesion_mask.nii.gz

# CSV file where SUV summary results will be written
OUTPUT_CSV = "/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Results/newResult 6.csv"

# Folder where new SUV-corrected PET DICOM images will be saved
SUV_OUTPUT_DIR = "/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images"

# Import tools for file and folder operations
import os

# Import math functions such as log and exp
import math

# Import CSV writer utilities
import csv

# Import copy so we can duplicate DICOM datasets safely before editing
import copy

# Import date and time handling utilities
from datetime import datetime, timedelta

# Import NumPy for arrays and numerical operations
import numpy as np

# Import pydicom for reading and writing DICOM medical image files
import pydicom

# Import pandas for tables and DataFrames
import pandas as pd

# Import binary erosion to shrink masks by a chosen number of voxels
from scipy.ndimage import binary_erosion

# Import nibabel to read NIfTI mask files (.nii.gz)
import nibabel as nib

# Import UID generator for creating new DICOM identifiers
from pydicom.uid import generate_uid


# Safely convert a value to float
def safe_float(x, default=None):
    try:
        # If the value is missing or empty, return the default
        if x is None or x == "":
            return default
        # Try converting to float
        return float(x)
    except Exception:
        # If conversion fails, return the default
        return default


# Parse a DICOM TM (time) string
def parse_dicom_tm(tm_str):
    # Return None if empty
    if not tm_str:
        return None

    # Convert to string and remove surrounding spaces
    s = str(tm_str).strip()

    # Default fractional seconds = 0
    frac = 0

    # If there is a decimal point, extract fractional seconds
    if "." in s:
        _, frac_str = s.split(".", 1)
        # Pad or trim to 6 digits for microseconds
        frac = int((frac_str + "000000")[:6])

    # Take the part before decimal and pad to HHMMSS
    base = s.split(".")[0].ljust(6, "0")

    try:
        # Parse hours
        h = int(base[0:2])

        # Parse minutes
        m = int(base[2:4])

        # Parse seconds
        sec = int(base[4:6])

        # Return time components plus microseconds
        return h, m, sec, frac
    except Exception:
        # Return None if parsing fails
        return None


# Parse a DICOM DA (date) string
def parse_dicom_da(da_str):
    # Return None if empty
    if not da_str:
        return None
    try:
        # Parse date in YYYYMMDD format
        return datetime.strptime(str(da_str), "%Y%m%d")
    except Exception:
        # Return None if invalid
        return None


# Combine separate DICOM date and time strings into one datetime
def combine_da_tm(date_str, time_str):
    # Parse the date
    d = parse_dicom_da(date_str)

    # Parse the time
    t = parse_dicom_tm(time_str)

    # If either failed, return None
    if d is None or t is None:
        return None

    # Replace time part of the date object and return full datetime
    return d.replace(hour=t[0], minute=t[1], second=t[2], microsecond=t[3])


# Parse a DICOM DT (datetime) string
def parse_dicom_dt(dt_str):
    # Return None if empty
    if not dt_str:
        return None

    # Convert to clean string
    s = str(dt_str).strip()

    # Remove timezone offset if present, because strptime below does not use it
    for sign in ["+", "-"]:
        if len(s) > 14 and sign in s[14:]:
            s = s.split(sign, 1)[0]
            break

    # Try several possible DICOM datetime formats
    for fmt in ["%Y%m%d%H%M%S.%f", "%Y%m%d%H%M%S", "%Y%m%d%H%M", "%Y%m%d"]:
        try:
            return datetime.strptime(s, fmt)
        except Exception:
            pass

    # If none worked, return None
    return None


# Read a private DICOM tag as datetime if present
def parse_private_dt(ds, tag):
    elem = ds.get(tag)
    if elem is None:
        return None
    return parse_dicom_dt(getattr(elem, "value", None))


# Find all case folders inside the root directory
def find_case_dirs(root_dir):
    return [
        # Build full path for each folder name
        os.path.join(root_dir, name)
        # Loop over sorted names in the root directory
        for name in sorted(os.listdir(root_dir))
        # Keep only directories
        if os.path.isdir(os.path.join(root_dir, name))
    ]


# Find all PET DICOM files in a PT folder
def find_dicom_files(pt_dir):
    # Start with an empty list
    dicoms = []

    # Walk through all subfolders and files
    for root, _, files in os.walk(pt_dir):
        for f in files:
            # Full file path
            path = os.path.join(root, f)
            try:
                # Read only metadata first for speed
                ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)

                # Keep only PET modality files
                if getattr(ds, "Modality", None) == "PT":
                    dicoms.append(path)
            except Exception:
                # Ignore unreadable files
                pass

    # Fail if no PET files found
    if not dicoms:
        raise FileNotFoundError(f"No PET DICOM files found in {pt_dir}")

    # Return list of PET DICOM file paths
    return dicoms


# Find one NIfTI mask file in a mask folder
def find_mask_file(mask_dir):
    # Allowed mask extensions
    candidates = []

    # Search through folder tree
    for root, _, files in os.walk(mask_dir):
        for f in files:
            lower = f.lower()

            # Keep .nii and .nii.gz files
            if lower.endswith(".nii") or lower.endswith(".nii.gz"):
                candidates.append(os.path.join(root, f))

    # Fail if none found
    if not candidates:
        raise FileNotFoundError(f"No NIfTI mask file found in {mask_dir}")

    # Sort for deterministic behavior
    candidates.sort()

    # Return the first mask file
    return candidates[0]


# Sort PET slices into the correct order
def sort_pet_slices(datasets):
    # Define sort key for each DICOM slice
    def key(ds):
        # Try acquisition datetime first
        acq_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))

        # If not present, combine separate acquisition date and time
        if acq_dt is None:
            acq_dt = combine_da_tm(
                getattr(ds, "AcquisitionDate", None),
                getattr(ds, "AcquisitionTime", None),
            )

        # Default z-position
        z = 0.0

        # Read image position if available
        ipp = getattr(ds, "ImagePositionPatient", None)
        if ipp is not None and len(ipp) >= 3:
            z = float(ipp[2])

        # Instance number as a backup sort criterion
        inst = safe_float(getattr(ds, "InstanceNumber", None), 0)

        # Return a tuple used for sorting
        return (acq_dt or datetime.min, z, inst)

    # Sort all datasets using the key above
    return sorted(datasets, key=key)


# Check that important metadata are consistent across the series
def assert_series_consistency(datasets):
    # Attributes that should not vary slice to slice
    attrs_to_check = [
        "Units",
        "DecayCorrection",
        "PatientWeight",
        "PatientSize",
        "PatientSex",
    ]

    # Check each attribute
    for attr in attrs_to_check:
        # Collect values as strings
        vals = [str(getattr(ds, attr, None)) for ds in datasets]

        # If more than one unique value exists, fail
        if len(set(vals)) > 1:
            raise ValueError(f"Inconsistent series attribute: {attr} -> {set(vals)}")


# Load the PET DICOM series into a 3D volume
def load_pet_series(pt_dir):
    # Find PET files
    dicom_files = find_dicom_files(pt_dir)

    # Read all PET files fully
    datasets = [pydicom.dcmread(p, force=True) for p in dicom_files]

    # Sort slices
    datasets = sort_pet_slices(datasets)

    # Validate consistent metadata
    assert_series_consistency(datasets)

    # Prepare containers
    arrays, slopes, intercepts = [], [], []

    # Loop through each slice
    for ds in datasets:
        # Read pixel array as float64
        pixel = ds.pixel_array.astype(np.float64)

        # Reject enhanced multiframe data
        if pixel.ndim != 2:
            raise NotImplementedError("Enhanced multiframe PET is not supported.")

        # Store raw pixel array
        arrays.append(pixel)

        # Store per-slice rescale slope
        slopes.append(safe_float(getattr(ds, "RescaleSlope", 1.0), 1.0))

        # Store per-slice rescale intercept
        intercepts.append(safe_float(getattr(ds, "RescaleIntercept", 0.0), 0.0))

    # Stack slices into 3D volume [z, y, x]
    volume = np.stack(arrays, axis=0)

    # Return metadata, raw volume, slopes, intercepts
    return (
        datasets,
        volume,
        np.asarray(slopes, dtype=np.float64),
        np.asarray(intercepts, dtype=np.float64),
    )


# Apply DICOM rescale equation to each slice
def apply_rescale(volume, slopes, intercepts):
    # Create empty output array
    out = np.empty_like(volume, dtype=np.float64)

    # Process each slice
    for z in range(volume.shape[0]):
        # Apply slope and intercept
        out[z] = volume[z] * slopes[z] + intercepts[z]

    # Return rescaled values
    return out


# Extract radiopharmaceutical info block from DICOM
def get_radiopharm_item(ds):
    # Get RadiopharmaceuticalInformationSequence
    seq = getattr(ds, "RadiopharmaceuticalInformationSequence", None)

    # Fail if missing
    if not seq or len(seq) == 0:
        raise ValueError("Missing RadiopharmaceuticalInformationSequence")

    # Use the first item
    return seq[0]


# Get patient weight in kg
def get_weight_kg(ds):
    # Read patient weight
    w = safe_float(getattr(ds, "PatientWeight", None))

    # Fail if missing or invalid
    if w is None or w <= 0:
        raise ValueError("Missing or zero PatientWeight")

    # Convert grams to kg if it looks too large
    if w > 1000:
        w /= 1000.0

    # Return kg
    return w


# Get patient weight in grams
def get_weight_g(ds):
    # Convert kg to grams
    return get_weight_kg(ds) * 1000.0


# Get patient height in meters
def get_height_m(ds):
    # Read patient height from PatientSize
    h = safe_float(getattr(ds, "PatientSize", None))

    # Fail if invalid
    if h is None or h <= 0:
        raise ValueError("Missing or invalid PatientSize")

    return h


# Get patient height in centimeters
def get_height_cm(ds):
    # Convert meters to cm
    return get_height_m(ds) * 100.0


# Normalize patient sex code
def get_patient_sex(ds):
    # Read and uppercase sex code
    sex = str(getattr(ds, "PatientSex", "O") or "O").upper()

    # Return only accepted values; otherwise use O
    return sex if sex in ["M", "F", "O"] else "O"


# Get injected radionuclide dose in becquerels
def get_total_dose_bq(ds):
    # Get radiopharmaceutical metadata block
    rp = get_radiopharm_item(ds)

    # Read total dose
    dose = safe_float(getattr(rp, "RadionuclideTotalDose", None))

    # Fail if missing or invalid
    if dose is None or dose <= 0:
        raise ValueError("Missing or invalid RadionuclideTotalDose")

    # If it looks too small, assume MBq and convert to Bq
    if dose < 1e7:
        dose *= 1e6

    return dose


# Get radionuclide half-life in seconds
def get_half_life_s(ds):
    # Get radiopharmaceutical metadata block
    rp = get_radiopharm_item(ds)

    # Read half-life
    hl = safe_float(getattr(rp, "RadionuclideHalfLife", None))

    # Fail if missing or invalid
    if hl is None or hl <= 0:
        raise ValueError("Missing or invalid RadionuclideHalfLife")

    return hl


# Determine injection start datetime
def get_radiopharm_start_datetime(ds):
    # Get radiopharmaceutical metadata block
    rp = get_radiopharm_item(ds)

    # Try full datetime first
    dt = parse_dicom_dt(getattr(rp, "RadiopharmaceuticalStartDateTime", None))
    if dt is not None:
        return dt

    # Otherwise try start time only
    start_time = getattr(rp, "RadiopharmaceuticalStartTime", None)
    if not start_time:
        raise ValueError("Missing radiopharmaceutical start date/time")

    # Use acquisition or series date/time as date anchor
    acq_date = getattr(ds, "AcquisitionDate", None) or getattr(ds, "SeriesDate", None)
    acq_time = getattr(ds, "AcquisitionTime", None) or getattr(ds, "SeriesTime", None)

    # Build start datetime using acquisition date
    start_dt = combine_da_tm(acq_date, start_time)

    # Build acquisition datetime
    acq_dt = combine_da_tm(acq_date, acq_time)

    # Fail if start datetime could not be built
    if start_dt is None:
        raise ValueError("Could not construct radiopharmaceutical start datetime")

    # If start appears later than acquisition, assume injection was the previous day
    if acq_dt is not None and start_dt > acq_dt:
        start_dt -= timedelta(days=1)

    return start_dt


# Compute average count-rate time correction
def average_count_rate_time_s(half_life_s, frame_duration_s):
    # Decay constant
    lam = math.log(2.0) / half_life_s

    # Return average count rate time
    return (1.0 / lam) * math.log((lam * frame_duration_s) / (1.0 - math.exp(-lam * frame_duration_s)))


# Compute reference time used for decay correction START
def get_reference_time_start(ds):
    # 1) Standard public attribute if present
    dc_dt = parse_dicom_dt(getattr(ds, "DecayCorrectionDateTime", None))
    if dc_dt is not None:
        return dc_dt

    # 2) GE private PET scan datetime (0009,100D)
    ge_dt = parse_private_dt(ds, (0x0009, 0x100D))
    if ge_dt is not None:
        return ge_dt

    # 3) Siemens private Decay Correction DateTime (0071,1022)
    siemens_dt = parse_private_dt(ds, (0x0071, 0x1022))
    if siemens_dt is not None:
        return siemens_dt

    # 4) Backcompute from acquisition datetime + frame timing
    half_life_s = get_half_life_s(ds)

    # Read frame duration in ms
    frame_duration_ms = safe_float(getattr(ds, "ActualFrameDuration", None))

    # Read frame reference time in ms
    frame_ref_ms = safe_float(getattr(ds, "FrameReferenceTime", None))

    # Try acquisition datetime directly
    acq_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))
    if acq_dt is None:
        # Fallback to separate acquisition date and time
        acq_dt = combine_da_tm(
            getattr(ds, "AcquisitionDate", None),
            getattr(ds, "AcquisitionTime", None)
        )

    # Fail if acquisition datetime missing
    if acq_dt is None:
        raise ValueError("Missing acquisition datetime")

    # Fail if frame timing metadata missing
    if frame_duration_ms is None or frame_ref_ms is None:
        raise ValueError("Missing ActualFrameDuration or FrameReferenceTime")

    # Convert ms to seconds
    frame_duration_s = frame_duration_ms / 1000.0
    frame_ref_s = frame_ref_ms / 1000.0

    # Compute average count-rate time
    tave_s = average_count_rate_time_s(half_life_s, frame_duration_s)

    # Return reference datetime
    return acq_dt + timedelta(seconds=(tave_s - frame_ref_s))


# Lean body mass using James formula variant
def lbm_james128_kg(weight_kg, height_cm, sex):
    male = 1.10 * weight_kg - 128.0 * (weight_kg / height_cm) ** 2
    female = 1.07 * weight_kg - 148.0 * (weight_kg / height_cm) ** 2
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


# Lean body mass using Morgan formula variant
def lbm_morgan_kg(weight_kg, height_cm, sex):
    male = 1.10 * weight_kg - 120.0 * (weight_kg / height_cm) ** 2
    female = 1.07 * weight_kg - 148.0 * (weight_kg / height_cm) ** 2
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


# Lean body mass using Janmahasatian formula
def lbm_janma_kg(weight_kg, height_cm, sex):
    bmi = weight_kg / ((height_cm / 100.0) ** 2)
    male = 9270.0 * weight_kg / (6680.0 + 216.0 * bmi)
    female = 9270.0 * weight_kg / (8780.0 + 244.0 * bmi)
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


# Ideal body weight formula
def ibw_kg(height_cm, sex):
    male = 48.0 + 1.06 * (height_cm - 152.0)
    female = 45.5 + 0.91 * (height_cm - 152.0)
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


# Body surface area, returned in cm^2
def bsa_cm2(weight_kg, height_cm):
    bsa_m2 = 0.007184 * (height_cm ** 0.725) * (weight_kg ** 0.425)
    return bsa_m2 * 10000.0


# Convert BQML PET values to SUVbw
def convert_bqml_to_suvbw(datasets, values):
    ds0 = datasets[0]
    weight_g = get_weight_g(ds0)
    dose_bq_admin = get_total_dose_bq(ds0)
    half_life_s = get_half_life_s(ds0)
    lam = math.log(2.0) / half_life_s
    decay_correction = str(getattr(ds0, "DecayCorrection", "") or "").upper().strip()

    if decay_correction == "ADMIN":
        return values * (weight_g / dose_bq_admin)

    if decay_correction == "START":
        tref = get_reference_time_start(ds0)
        tadm = get_radiopharm_start_datetime(ds0)
        delta_s = (tref - tadm).total_seconds()
        dose_at_ref = dose_bq_admin * math.exp(-lam * delta_s)
        return values * (weight_g / dose_at_ref)

    if decay_correction == "NONE":
        tadm = get_radiopharm_start_datetime(ds0)
        out = np.empty_like(values, dtype=np.float64)

        for z, ds in enumerate(datasets):
            acq_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))
            if acq_dt is None:
                acq_dt = combine_da_tm(
                    getattr(ds, "AcquisitionDate", None),
                    getattr(ds, "AcquisitionTime", None)
                )

            if acq_dt is None:
                raise ValueError("Missing acquisition time for DecayCorrection=NONE")

            frame_duration_ms = safe_float(getattr(ds, "ActualFrameDuration", None))
            if frame_duration_ms is None:
                raise ValueError("Missing ActualFrameDuration for DecayCorrection=NONE")

            T = frame_duration_ms / 1000.0
            factor = (
                weight_g / dose_bq_admin
                * (lam * T) / (1.0 - math.exp(-lam * T))
                * math.exp(lam * (acq_dt - tadm).total_seconds())
            )
            out[z] = values[z] * factor

        return out

    raise NotImplementedError(f"Unsupported DecayCorrection: {decay_correction}")


# Convert GML values to SUVbw
def convert_gml_to_suvbw(ds0, values):
    suv_type = str(getattr(ds0, "SUVType", "") or "").upper().strip()
    if suv_type == "":
        suv_type = "BW"

    if suv_type == "BW":
        return values

    weight_kg = get_weight_kg(ds0)
    height_cm = get_height_cm(ds0)
    sex = get_patient_sex(ds0)

    if suv_type == "LBM":
        factor = lbm_morgan_kg(weight_kg, height_cm, sex)
    elif suv_type == "LBMJAMES128":
        factor = lbm_james128_kg(weight_kg, height_cm, sex)
    elif suv_type == "LBMJANMA":
        factor = lbm_janma_kg(weight_kg, height_cm, sex)
    elif suv_type == "IBW":
        factor = ibw_kg(height_cm, sex)
    else:
        raise NotImplementedError(f"Unsupported SUVType in GML: {suv_type}")

    if factor <= 0:
        raise ValueError(f"Computed non-positive normalization factor for SUVType={suv_type}")

    return values * (weight_kg / factor)


# Convert CM2ML values to SUVbw
def convert_cm2ml_to_suvbw(ds0, values):
    suv_type = str(getattr(ds0, "SUVType", "") or "").upper().strip()

    # Allow missing SUVType, but if present it should be BSA
    if suv_type not in ["", "BSA"]:
        raise ValueError(f"Unexpected SUVType for CM2ML: {suv_type}")

    weight_kg = get_weight_kg(ds0)
    weight_g = weight_kg * 1000.0
    height_cm = get_height_cm(ds0)
    bsa = bsa_cm2(weight_kg, height_cm)

    if bsa <= 0:
        raise ValueError("Computed non-positive BSA")

    return values * (weight_g / bsa)


# Convert Philips CNTS values to SUVbw
def convert_cnts_to_suvbw(datasets, values):
    ds0 = datasets[0]
    suv_sf_tag = ds0.get((0x7053, 0x1000))
    ac_sf_tag = ds0.get((0x7053, 0x1009))

    suv_sf = safe_float(suv_sf_tag.value if suv_sf_tag is not None else None)
    ac_sf = safe_float(ac_sf_tag.value if ac_sf_tag is not None else None)

    if suv_sf is not None and suv_sf != 0:
        return values * suv_sf

    if ac_sf is not None and ac_sf != 0:
        bqml = values * ac_sf
        return convert_bqml_to_suvbw(datasets, bqml)

    raise NotImplementedError("CNTS without Philips scale factors is not supported.")


# Main converter that sends PET values to SUVbw depending on DICOM Units
def convert_to_suvbw(datasets, volume, slopes, intercepts):
    ds0 = datasets[0]

    if np.any(np.abs(intercepts) > 1e-12):
        raise ValueError("RescaleIntercept is not zero. Excluding dataset.")

    values = apply_rescale(volume, slopes, intercepts)
    units = str(getattr(ds0, "Units", "") or "").upper().strip()

    if units == "BQML":
        return convert_bqml_to_suvbw(datasets, values)
    if units == "GML":
        return convert_gml_to_suvbw(ds0, values)
    if units == "CM2ML":
        return convert_cm2ml_to_suvbw(ds0, values)
    if units == "CNTS":
        return convert_cnts_to_suvbw(datasets, values)

    raise NotImplementedError(f"Unsupported Units: {units}")


# Load a NIfTI mask and align it to PET shape
def load_nifti_mask(mask_path, pet_shape=None, erode_voxels=0):
    # Load the NIfTI image from disk
    nii = nib.load(mask_path)

    # Read voxel data as float and convert to boolean mask
    mask = nii.get_fdata() > 0

    # NIfTI masks often come as [x, y, z], while PET volume is [z, y, x]
    if pet_shape is not None:
        # Try common axis orders
        candidates = [
            mask,
            np.transpose(mask, (2, 1, 0)),
            np.transpose(mask, (2, 0, 1)),
            np.transpose(mask, (1, 0, 2)),
            np.transpose(mask, (1, 2, 0)),
            np.transpose(mask, (0, 2, 1)),
        ]

        aligned = None
        for cand in candidates:
            if cand.shape == pet_shape:
                aligned = cand
                break

        if aligned is None:
            raise ValueError(f"NIfTI mask shape mismatch: PET {pet_shape} vs mask {mask.shape}")
    else:
        aligned = mask

    # Optionally erode mask inward
    if erode_voxels > 0:
        aligned = binary_erosion(aligned, iterations=erode_voxels)

    return aligned


# Compute SUV summary metrics inside the mask
def calculate_suv_values(suv_volume, mask):
    if suv_volume.shape != mask.shape:
        raise ValueError(f"Shape mismatch: PET {suv_volume.shape} vs mask {mask.shape}")

    vals = suv_volume[mask]

    if vals.size == 0:
        raise ValueError("Mask is empty")

    return {
        "Nvox": int(vals.size),
        "SUVmin": round(float(np.min(vals)), 2),
        "SUVmedian": round(float(np.median(vals)), 2),
        "SUVmax": round(float(np.max(vals)), 2),
        "SUVmean": round(float(np.mean(vals)), 2),
    }


# Extract a small metadata summary for reporting
def summarize_case_metadata(ds):
    rp = get_radiopharm_item(ds)
    return {
        "Units": str(getattr(ds, "Units", "") or ""),
        "SUVType": str(getattr(ds, "SUVType", "") or ""),
        "DecayCorrection": str(getattr(ds, "DecayCorrection", "") or ""),
        "PatientWeight": safe_float(getattr(ds, "PatientWeight", None)),
        "PatientSize": safe_float(getattr(ds, "PatientSize", None)),
        "RadionuclideTotalDose": safe_float(getattr(rp, "RadionuclideTotalDose", None)),
        "RadionuclideHalfLife": safe_float(getattr(rp, "RadionuclideHalfLife", None)),
    }


# Process one case folder from start to finish
def process_case(case_dir, erode_voxels=0):
    # Extract case folder name
    case_name = os.path.basename(case_dir.rstrip(os.sep))

    # Build PET folder path
    pt_dir = os.path.join(case_dir, "PT")

    # Build mask folder path
    mask_dir = os.path.join(case_dir, "mask")

    # Find mask file
    mask_path = find_mask_file(mask_dir)

    # Load PET series
    datasets, volume, slopes, intercepts = load_pet_series(pt_dir)

    # Convert PET values to SUVbw
    suv = convert_to_suvbw(datasets, volume, slopes, intercepts)

    # Load and align NIfTI mask
    mask = load_nifti_mask(
        mask_path,
        pet_shape=suv.shape,
        erode_voxels=erode_voxels
    )

    # Compute SUV metrics inside the mask
    metrics = calculate_suv_values(suv, mask)

    # Collect metadata summary
    meta = summarize_case_metadata(datasets[0])

    # Return one output row
    return {
        "case": case_name,
        "mask_file": os.path.basename(mask_path),
        "erode_voxels": erode_voxels,
        "Nvox": metrics["Nvox"],
        "Units": meta["Units"],
        "SUVType": meta["SUVType"],
        "DecayCorrection": meta["DecayCorrection"],
        "SUVmin": metrics["SUVmin"],
        "SUVmedian": metrics["SUVmedian"],
        "SUVmax": metrics["SUVmax"],
        "SUVmean": metrics["SUVmean"],
    }


# Process every case folder and save summary CSV
def run_all_cases(root_dir, output_csv, erode_voxels=0):
    results = []

    # Make sure output folder exists
    out_parent = os.path.dirname(output_csv)
    if out_parent:
        os.makedirs(out_parent, exist_ok=True)

    for case_dir in find_case_dirs(root_dir):
        case_name = os.path.basename(case_dir)

        try:
            row = process_case(case_dir, erode_voxels=erode_voxels)
            results.append(row)
            print(
                f"{case_name}: "
                f"Units={row['Units']}, "
                f"DecayCorrection={row['DecayCorrection']}, "
                f"Nvox={row['Nvox']}, "
                f"SUVmin={row['SUVmin']:.2f}, "
                f"SUVmedian={row['SUVmedian']:.2f}, "
                f"SUVmax={row['SUVmax']:.2f}"
            )
        except Exception as e:
            print(f"{case_name}: FAILED -> {e}")

    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "case", "mask_file", "erode_voxels", "Nvox",
                "Units", "SUVType", "DecayCorrection",
                "SUVmin", "SUVmedian", "SUVmax", "SUVmean"
            ]
        )
        writer.writeheader()
        writer.writerows(results)

    return pd.DataFrame(results)


# Save a new DICOM PET series whose values represent SUVbw
def save_suv_dicom_series(case_dir, output_root):
    case_name = os.path.basename(case_dir.rstrip(os.sep))
    pt_dir = os.path.join(case_dir, "PT")

    datasets, volume, slopes, intercepts = load_pet_series(pt_dir)
    suv_volume = convert_to_suvbw(datasets, volume, slopes, intercepts)

    case_out_dir = os.path.join(output_root, case_name)
    os.makedirs(case_out_dir, exist_ok=True)

    suv_max = float(np.max(suv_volume))
    if suv_max <= 0:
        common_slope = 1.0
    else:
        common_slope = max(suv_max / 65535.0, 1e-6)

    new_series_uid = generate_uid()

    for z, ds in enumerate(datasets):
        new_ds = copy.deepcopy(ds)

        suv_slice = suv_volume[z].astype(np.float64)
        stored = np.round(suv_slice / common_slope)
        stored = np.clip(stored, 0, 65535).astype(np.uint16)

        new_ds.PixelData = stored.tobytes()
        new_ds.Rows, new_ds.Columns = stored.shape
        new_ds.BitsAllocated = 16
        new_ds.BitsStored = 16
        new_ds.HighBit = 15
        new_ds.PixelRepresentation = 0
        new_ds.SamplesPerPixel = 1
        new_ds.PhotometricInterpretation = "MONOCHROME2"

        new_ds.SeriesInstanceUID = new_series_uid
        new_ds.SOPInstanceUID = generate_uid()
        if hasattr(new_ds, "file_meta") and new_ds.file_meta is not None:
            new_ds.file_meta.MediaStorageSOPInstanceUID = new_ds.SOPInstanceUID

        new_ds.SeriesDescription = "SUVbw Corrected PET"
        new_ds.ImageType = ["DERIVED", "SECONDARY"]
        new_ds.Units = "GML"
        new_ds.SUVType = "BW"
        new_ds.RescaleSlope = float(common_slope)
        new_ds.RescaleIntercept = 0.0
        new_ds.SeriesNumber = int(getattr(ds, "SeriesNumber", 0) or 0) + 1000
        new_ds.InstanceNumber = z + 1

        out_path = os.path.join(case_out_dir, f"SUV_{z+1:04d}.dcm")
        new_ds.save_as(out_path, write_like_original=False)

    return case_name, case_out_dir, suv_volume.shape, common_slope

# Loop through everything inside the root directory in sorted order
for name in sorted(os.listdir(ROOT_DIR)):
    # Build the full path for this item
    case_path = os.path.join(ROOT_DIR, name)

    # Only continue if the item is a folder
    if os.path.isdir(case_path):
        # Print the case folder name and the files/folders inside it
        print(name, "->", os.listdir(case_path))

# Pick the first case folder found in ROOT_DIR
case_dir = find_case_dirs(ROOT_DIR)[0]

# Build the mask folder path
mask_dir = os.path.join(case_dir, "mask")

# Find the first NIfTI mask file
mask_path = find_mask_file(mask_dir)

# Load the NIfTI file
nii = nib.load(mask_path)

# Print file name, shape, and affine matrix
print("Mask file:", mask_path)
print("Mask shape:", nii.shape)
print("Affine:\n", nii.affine)

# Run the full SUV summary pipeline across all cases
df = run_all_cases(
    ROOT_DIR,          # folder containing all case folders
    OUTPUT_CSV,        # where to save the CSV summary
    erode_voxels=1     # shrink the mask by 1 voxel before measuring
)

# Display the result table
df

# Create the output folder for SUV-corrected PET DICOM series if it does not exist
os.makedirs(SUV_OUTPUT_DIR, exist_ok=True)

# Prepare a list to store per-case save summaries
saved_cases = []

# Loop through all case folders
for case_dir in find_case_dirs(ROOT_DIR):
    # Extract just the case folder name
    case_name = os.path.basename(case_dir)

    try:
        # Convert PET to SUVbw and save it as a new DICOM series
        name, out_dir, shape, slope = save_suv_dicom_series(case_dir, SUV_OUTPUT_DIR)

        # Save summary info for this case
        saved_cases.append({
            "case": name,
            "output_folder": out_dir,
            "shape": str(shape),
            "rescale_slope": slope
        })

        # Print success message
        print(f"{name}: SAVED -> {out_dir} | shape={shape} | RescaleSlope={slope:.8f}")

    except Exception as e:
        # Print failure message and continue with the next case
        print(f"{case_name}: FAILED -> {e}")

# Convert summary list to a DataFrame
saved_df = pd.DataFrame(saved_cases).sort_values("case").reset_index(drop=True)

# Display saved-case summary table
saved_df

validation_01 -> ['attribution_ACRIN-FLT-Breast.md', 'mask', 'PT']
validation_02 -> ['attribution_ACRIN-FLT-Breast.md', 'mask', 'PT']
validation_03 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_04 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_05 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_06 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_07 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_08 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_09 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_10 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_11 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_12 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_13 -> ['attribution_QIN-BREAST.md', 'mask', 'PT']
validation_14 -> ['attribution_NSCLC-Radiogenomics.md', 'PT', 'mask']
validation_15 -> ['attribution_Soft

validation_01: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images/validation_01 | shape=(45, 144, 144) | RescaleSlope=0.00073086


validation_02: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images/validation_02 | shape=(255, 144, 144) | RescaleSlope=0.00405916


validation_03: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images/validation_03 | shape=(205, 128, 128) | RescaleSlope=0.00000100


validation_04: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images/validation_04 | shape=(172, 144, 144) | RescaleSlope=0.00045434


validation_05: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images/validation_05 | shape=(239, 128, 128) | RescaleSlope=0.00110775
validation_06: FAILED -> Missing or zero PatientWeight


... (3 outputs left out)

validation_13: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images/validation_13 | shape=(83, 128, 128) | RescaleSlope=0.00000100
validation_14: FAILED -> CNTS without Philips scale factors is not supported.


validation_15: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images/validation_15 | shape=(267, 128, 128) | RescaleSlope=0.00028144


validation_16: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/SUV_Corrected PET_Images/validation_16 | shape=(324, 168, 168) | RescaleSlope=0.00024529


,case,output_folder,shape,rescale_slope
0,validation_01,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(45, 144, 144)",0.000731
1,validation_02,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(255, 144, 144)",0.004059
2,validation_03,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(205, 128, 128)",0.000001
3,validation_04,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(172, 144, 144)",0.000454
4,validation_05,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(239, 128, 128)",0.001108
5,validation_07,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(246, 128, 128)",0.000001
6,validation_09,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(213, 144, 144)",0.001052
7,validation_10,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(195, 128, 128)",0.000269
8,validation_13,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(83, 128, 128)",0.000001
9,validation_15,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(267, 128, 128)",0.000281
